# Snowflake Connection Diagnostics

Use this notebook to validate Snowflake connectivity with the official Python connector and key pair authentication.

This notebook:
- Reads the same `.env` file used by the main pipeline notebook
- Tests `SNOWFLAKE_ACCOUNT_1` only
- Verifies connector version, account identifier format, private key path, DNS, and TCP reachability
- Attempts a connector login with `authenticator="SNOWFLAKE_JWT"`
- Verifies Trust Center access with a minimal read query
- Prints actionable troubleshooting guidance for common failure modes

This notebook does not send any data to Google SecOps and does not modify Snowflake.


In [ ]:
# Cell 2 - Install dependencies
%pip install --quiet "snowflake-connector-python[pandas]>=3.16.0" "python-dotenv>=1.0.0" "requests>=2.31.0" "packaging>=24.0"
print("Dependencies installed.")

In [ ]:
# Cell 3 - Helpers and diagnostics logic
import importlib.metadata
import os
import platform
import socket
import textwrap
from pathlib import Path
from typing import Callable, Dict, List, Optional, Tuple

from dotenv import load_dotenv
from packaging.version import Version
import snowflake.connector

NOTEBOOK_FILENAME = "snowflake_connection_diagnostics.ipynb"
MIN_CONNECTOR_VERSION = Version("3.16.0")
STATUS_PASS = "PASS"
STATUS_WARN = "WARN"
STATUS_FAIL = "FAIL"

KEY_PAIR_POLICY_SQL = """DESC USER <your_user>;
SHOW PARAMETERS LIKE 'NETWORK_POLICY' IN ACCOUNT;
SHOW PARAMETERS LIKE 'NETWORK_POLICY' IN USER <your_user>;
SHOW AUTHENTICATION POLICIES ON ACCOUNT;
SHOW AUTHENTICATION POLICIES ON USER <your_user>;
DESC AUTHENTICATION POLICY <database>.<schema>.<policy_name>;"""


class ConfigError(RuntimeError):
    """Raised when the notebook configuration is incomplete or invalid."""


def build_result(
    status: str,
    summary: str,
    details: Optional[List[str]] = None,
    actions: Optional[List[str]] = None,
    raw_error: Optional[str] = None,
) -> Dict[str, object]:
    """Create a uniform diagnostics record."""
    return {
        "status": status,
        "summary": summary,
        "details": details or [],
        "actions": actions or [],
        "raw_error": raw_error,
    }


def skipped_result(summary: str) -> Dict[str, object]:
    """Create a skipped-stage record."""
    return build_result(
        STATUS_WARN,
        summary,
        actions=["Resolve the earlier failure and rerun the notebook."],
    )


def mask_secret(secret: str) -> str:
    """Mask a secret so it can be displayed safely in notebook output."""
    if not secret:
        return "<empty>"
    if len(secret) <= 8:
        return "*" * len(secret)
    return f"{secret[:4]}...{secret[-4:]}"


def detect_project_dir() -> Path:
    """Resolve the notebook directory or fall back to the current working directory."""
    override = os.getenv("NOTEBOOK_ROOT")
    if override:
        return Path(override).expanduser().resolve()

    search_roots = [Path.cwd(), *Path.cwd().parents]
    for root in search_roots:
        if (root / NOTEBOOK_FILENAME).exists():
            return root.resolve()
    return Path.cwd().resolve()


def get_env_path(project_dir: Optional[Path] = None) -> Path:
    """Resolve the configuration file path."""
    project_dir = project_dir or detect_project_dir()
    env_path = os.getenv("CONFIG_ENV_PATH", str(project_dir / ".env"))
    return Path(env_path).expanduser().resolve()


def normalize_private_key_path(raw_path: str) -> Path:
    """Resolve the private key path for diagnostics and connector use."""
    return Path(raw_path).expanduser().resolve()


def load_config() -> Dict[str, object]:
    """Load SNOWFLAKE_ACCOUNT_1 configuration from the notebook environment."""
    project_dir = detect_project_dir()
    env_path = get_env_path(project_dir)
    load_dotenv(env_path, override=False)

    required_keys = [
        "SNOWFLAKE_ACCOUNT_1",
        "SNOWFLAKE_USER_1",
        "SNOWFLAKE_PRIVATE_KEY_PATH_1",
    ]
    missing = [key for key in required_keys if not os.getenv(key, "").strip()]
    if missing:
        missing_lines = "\n".join(f"- {key}" for key in missing)
        raise ConfigError(
            "Missing required Snowflake configuration values.\n"
            f"Config file: {env_path}\n"
            f"Missing keys:\n{missing_lines}\n\n"
            "Copy .env.example to .env and fill in SNOWFLAKE_ACCOUNT_1, "
            "SNOWFLAKE_USER_1, and SNOWFLAKE_PRIVATE_KEY_PATH_1 before rerunning this notebook."
        )

    return {
        "project_dir": project_dir,
        "env_path": env_path,
        "env_found": env_path.exists(),
        "account": os.getenv("SNOWFLAKE_ACCOUNT_1", "").strip(),
        "user": os.getenv("SNOWFLAKE_USER_1", "").strip(),
        "private_key_path": str(
            normalize_private_key_path(os.getenv("SNOWFLAKE_PRIVATE_KEY_PATH_1", "").strip())
        ),
        "private_key_passphrase": os.getenv(
            "SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_1", ""
        ),
        "role": os.getenv("SNOWFLAKE_ROLE_1", "TRUST_CENTER_VIEWER").strip()
        or "TRUST_CENTER_VIEWER",
        "warehouse": os.getenv("SNOWFLAKE_WAREHOUSE_1", "COMPUTE_WH").strip()
        or "COMPUTE_WH",
        "label": os.getenv("SNOWFLAKE_LABEL_1", "account-1").strip() or "account-1",
    }


def prepare_config() -> Tuple[Optional[Dict[str, object]], Dict[str, object]]:
    """Load configuration and return an env diagnostics record."""
    try:
        config = load_config()
    except Exception as exc:
        env_path = get_env_path()
        return None, build_result(
            STATUS_FAIL,
            "Configuration is incomplete. The notebook stopped before any network calls.",
            details=[
                f"Config file: {env_path}",
                "Required keys: SNOWFLAKE_ACCOUNT_1, SNOWFLAKE_USER_1, SNOWFLAKE_PRIVATE_KEY_PATH_1",
                "Expected behavior: fail fast with no interactive prompts.",
            ],
            actions=[
                "Copy .env.example to .env if the file does not exist yet.",
                "Fill in SNOWFLAKE_ACCOUNT_1, SNOWFLAKE_USER_1, and SNOWFLAKE_PRIVATE_KEY_PATH_1.",
                "Set SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_1 when the key file is encrypted.",
                "Keep SNOWFLAKE_ROLE_1 and SNOWFLAKE_WAREHOUSE_1 aligned with the role you want to test.",
            ],
            raw_error=f"{type(exc).__name__}: {exc}",
        )

    return config, build_result(
        STATUS_PASS,
        "Snowflake configuration loaded successfully.",
        details=[
            f"Project dir: {config['project_dir']}",
            f"Env path: {config['env_path']} ({'found' if config['env_found'] else 'not found; using process environment'})",
            f"Account label: {config['label']}",
            f"Account identifier: {config['account']}",
            f"User: {config['user']}",
            f"Role: {config['role']}",
            f"Warehouse: {config['warehouse']}",
            f"Private key path: {config['private_key_path']}",
            f"Private key passphrase: {'set' if config['private_key_passphrase'] else 'not set'}",
        ],
    )


def validate_account_identifier(account: str) -> Optional[Dict[str, object]]:
    """Validate that the account value is an account identifier, not a full URL."""
    lowered = account.lower().strip()
    if not lowered:
        return build_result(
            STATUS_FAIL,
            "Snowflake account identifier is blank.",
            actions=["Set SNOWFLAKE_ACCOUNT_1 in .env and rerun the notebook."],
        )

    if lowered.startswith("http://") or lowered.startswith("https://") or "/" in lowered or "snowflakecomputing.com" in lowered:
        return build_result(
            STATUS_FAIL,
            "Snowflake account identifier format looks wrong.",
            details=[
                f"Received value: {account}",
                "This field must be an account identifier, not a full hostname or URL.",
            ],
            actions=[
                "Use values like xy12345, xy12345.us-east-1, xy12345.east-us-2.azure, or myorg-myacct.",
                "Do not include https:// or .snowflakecomputing.com in SNOWFLAKE_ACCOUNT_1.",
            ],
        )
    return None


def validate_private_key_path(private_key_path: str) -> Optional[Dict[str, object]]:
    """Validate that the configured private key path exists and points to a file."""
    if not private_key_path:
        return build_result(
            STATUS_FAIL,
            "Snowflake private key path is blank.",
            actions=["Set SNOWFLAKE_PRIVATE_KEY_PATH_1 in .env and rerun the notebook."],
        )

    path = normalize_private_key_path(private_key_path)
    if not path.exists():
        return build_result(
            STATUS_FAIL,
            "Snowflake private key file was not found.",
            details=[f"Configured path: {path}"],
            actions=[
                "Confirm the file exists on this machine.",
                "Update SNOWFLAKE_PRIVATE_KEY_PATH_1 to point to the correct PKCS#8 private key file.",
            ],
        )
    if not path.is_file():
        return build_result(
            STATUS_FAIL,
            "Snowflake private key path is not a file.",
            details=[f"Configured path: {path}"],
            actions=["Point SNOWFLAKE_PRIVATE_KEY_PATH_1 to a readable private key file."],
        )
    return None


def derive_hostname(account: str) -> str:
    """Build the default Snowflake hostname from the account identifier."""
    return f"{account}.snowflakecomputing.com"


def get_connector_version() -> str:
    """Return the installed Snowflake connector version."""
    try:
        return importlib.metadata.version("snowflake-connector-python")
    except importlib.metadata.PackageNotFoundError:
        return getattr(snowflake.connector, "__version__", "unknown")


def resolve_dns_addresses(hostname: str) -> List[str]:
    """Resolve DNS addresses for the Snowflake hostname."""
    infos = socket.getaddrinfo(hostname, None, proto=socket.IPPROTO_TCP)
    addresses = sorted({info[4][0] for info in infos})
    if not addresses:
        raise RuntimeError(f"DNS lookup returned no addresses for {hostname}")
    return addresses


def check_tcp_connectivity(hostname: str, port: int = 443, timeout: int = 5) -> None:
    """Open and close a TCP socket to verify basic reachability."""
    with socket.create_connection((hostname, port), timeout=timeout):
        return None


def classify_exception(exc: Exception, stage: str) -> Dict[str, object]:
    """Map raw exceptions to actionable troubleshooting guidance."""
    raw_error = f"{type(exc).__name__}: {exc}"
    lowered = raw_error.lower()

    network_patterns = [
        "250001",
        "could not connect to snowflake backend",
        "timed out",
        "timeout",
        "ssl",
        "proxy",
        "name resolution",
        "getaddrinfo",
        "connection refused",
        "connection reset",
        "failed to establish a new connection",
        "nodename nor servname provided",
        "temporary failure in name resolution",
        "network is unreachable",
    ]
    auth_patterns = [
        "jwt",
        "private key",
        "could not deserialize key data",
        "bad decrypt",
        "pkcs8",
        "password was given but private key is not encrypted",
        "failed to load private key",
        "incorrect username or password",
        "authentication",
    ]
    grants_patterns = [
        "trust_center",
        "insufficient privileges",
        "not authorized",
        "does not exist or not authorized",
        "schema not found",
        "permission",
    ]

    if any(pattern in lowered for pattern in network_patterns):
        return build_result(
            STATUS_FAIL,
            "Snowflake service connectivity failed before the notebook could complete the requested stage.",
            details=[
                f"Stage: {stage}",
                "Classification: Network / Allowlist",
                "Typical causes: firewall rules, proxy settings, SSL inspection, VPN routing, or blocked OCSP/network paths.",
            ],
            actions=[
                "Confirm the derived Snowflake hostname resolves and outbound TCP/443 is open from this machine.",
                "If your company uses a proxy or TLS inspection appliance, bypass Snowflake domains. SSL certificate interception breaks Snowflake clients.",
                "Ask the network team to allow the hostnames returned by SYSTEM$ALLOWLIST() or SYSTEM$ALLOWLIST_PRIVATELINK().",
                "Snowflake OCSP checks can require outbound TCP/80 in addition to TCP/443.",
                "Run SnowCD from the same workstation or runner to validate the network path end to end.",
            ],
            raw_error=raw_error,
        )

    if any(pattern in lowered for pattern in auth_patterns):
        return build_result(
            STATUS_FAIL,
            "Snowflake rejected the key pair login or the private key could not be loaded.",
            details=[
                f"Stage: {stage}",
                "Classification: Key Pair / Auth Policy",
                "Typical causes: wrong private key path, wrong passphrase, mismatched public key, or an authentication policy that blocks SNOWFLAKE_JWT.",
            ],
            actions=[
                "Confirm the private key file matches the public key assigned to the configured Snowflake user.",
                "If the private key is encrypted, confirm SNOWFLAKE_PRIVATE_KEY_PASSPHRASE_1 is correct.",
                "Ask a Snowflake admin to run the following and compare the configured key fingerprint and auth policy:",
                textwrap.dedent(KEY_PAIR_POLICY_SQL).strip(),
            ],
            raw_error=raw_error,
        )

    if any(pattern in lowered for pattern in grants_patterns):
        return build_result(
            STATUS_FAIL,
            "The connector login may have succeeded, but the configured role could not access Snowflake Trust Center.",
            details=[
                f"Stage: {stage}",
                "Classification: Edition / Grants / Role",
                "Typical causes: Trust Center is not enabled, the account edition does not support it, or the role lacks TRUST_CENTER_VIEWER access.",
            ],
            actions=[
                "Confirm the Snowflake account edition supports Trust Center and that the feature is enabled.",
                "Grant the TRUST_CENTER_VIEWER database role to the role used in this notebook, or test with ACCOUNTADMIN if appropriate.",
                "Verify that SNOWFLAKE_ROLE_1 and SNOWFLAKE_WAREHOUSE_1 are valid for the configured user.",
            ],
            raw_error=raw_error,
        )

    return build_result(
        STATUS_FAIL,
        "The notebook hit an unexpected Snowflake error that did not match a known diagnostics category.",
        details=[f"Stage: {stage}", "Classification: Unclassified"],
        actions=[
            "Review the raw error below.",
            "If the error is network-related, inspect proxy, firewall, SSL inspection, and allowlist configuration.",
            "If the error is authentication-related, inspect key pair configuration and authentication policies.",
        ],
        raw_error=raw_error,
    )


def run_preflight_checks(
    config: Dict[str, object],
    connector_version_override: Optional[str] = None,
    dns_resolver: Optional[Callable[[str], List[str]]] = None,
    tcp_checker: Optional[Callable[[str, int, int], None]] = None,
) -> Dict[str, object]:
    """Validate local prerequisites before opening a Snowflake connector session."""
    identifier_result = validate_account_identifier(str(config["account"]))
    key_path_result = validate_private_key_path(str(config["private_key_path"]))
    python_version = platform.python_version()
    connector_version = connector_version_override or get_connector_version()
    hostname = derive_hostname(str(config["account"]))

    if identifier_result is not None:
        identifier_result["details"] = [
            f"Python version: {python_version}",
            f"Connector version: {connector_version}",
            *identifier_result.get("details", []),
        ]
        return identifier_result

    if key_path_result is not None:
        key_path_result["details"] = [
            f"Python version: {python_version}",
            f"Connector version: {connector_version}",
            *key_path_result.get("details", []),
        ]
        return key_path_result

    if Version(connector_version) < MIN_CONNECTOR_VERSION:
        return build_result(
            STATUS_FAIL,
            "The installed Snowflake connector is older than the notebook baseline.",
            details=[
                f"Python version: {python_version}",
                f"Installed connector version: {connector_version}",
                f"Required connector version: >= {MIN_CONNECTOR_VERSION}",
            ],
            actions=[
                "Upgrade the connector with %pip install --upgrade 'snowflake-connector-python[pandas]>=3.16.0'.",
                "Restart the kernel after the upgrade so the new connector version is loaded.",
            ],
        )

    dns_resolver = dns_resolver or resolve_dns_addresses
    tcp_checker = tcp_checker or check_tcp_connectivity

    try:
        addresses = dns_resolver(hostname)
        tcp_checker(hostname, 443, 5)
    except Exception as exc:
        result = classify_exception(exc, "preflight")
        result["details"] = [
            f"Python version: {python_version}",
            f"Connector version: {connector_version}",
            f"Derived hostname: {hostname}",
            f"Private key path: {config['private_key_path']}",
            *result.get("details", []),
        ]
        return result

    return build_result(
        STATUS_PASS,
        "Local prerequisites look good and the Snowflake hostname is reachable from this machine.",
        details=[
            f"Python version: {python_version}",
            f"Connector version: {connector_version}",
            f"Derived hostname: {hostname}",
            f"Private key path: {config['private_key_path']}",
            f"Private key passphrase: {'set' if config['private_key_passphrase'] else 'not set'}",
            f"DNS addresses: {', '.join(addresses)}",
            "TCP 443 reachability: success",
        ],
    )


def connect_with_key_pair(
    config: Dict[str, object],
    connect_fn: Optional[Callable[..., object]] = None,
):
    """Open a Snowflake session with the official connector and key pair authentication."""
    connect_fn = connect_fn or snowflake.connector.connect
    return connect_fn(
        account=config["account"],
        user=config["user"],
        authenticator="SNOWFLAKE_JWT",
        private_key_file=config["private_key_path"],
        private_key_file_pwd=config["private_key_passphrase"] or None,
        role=config["role"],
        warehouse=config["warehouse"],
        client_session_keep_alive=False,
        network_timeout=30,
    )


def run_login_check(
    config: Dict[str, object],
    connect_fn: Optional[Callable[..., object]] = None,
) -> Tuple[Dict[str, object], Optional[object]]:
    """Attempt a connector login and return both the status record and connection object."""
    try:
        conn = connect_with_key_pair(config, connect_fn=connect_fn)
    except Exception as exc:
        return classify_exception(exc, "login"), None

    return build_result(
        STATUS_PASS,
        "The official Snowflake Python connector established a session with key pair authentication.",
        details=[
            "Authenticator: SNOWFLAKE_JWT",
            f"Account identifier: {config['account']}",
            f"Role requested: {config['role']}",
            f"Warehouse requested: {config['warehouse']}",
            f"Private key path: {config['private_key_path']}",
            "Network timeout: 30 seconds",
        ],
    ), conn


def run_access_checks(conn: object) -> Dict[str, object]:
    """Validate basic session context plus Trust Center schema visibility and read access."""
    session_details: List[str] = []
    try:
        with conn.cursor() as cur:
            cur.execute(
                "SELECT CURRENT_ACCOUNT(), CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_VERSION()"
            )
            current_account, current_user, current_role, current_warehouse, current_version = cur.fetchone()
            session_details = [
                f"CURRENT_ACCOUNT(): {current_account}",
                f"CURRENT_USER(): {current_user}",
                f"CURRENT_ROLE(): {current_role}",
                f"CURRENT_WAREHOUSE(): {current_warehouse}",
                f"CURRENT_VERSION(): {current_version}",
            ]

            cur.execute("SHOW SCHEMAS IN DATABASE SNOWFLAKE LIKE 'TRUST_CENTER'")
            schemas = cur.fetchall()
            if not schemas:
                raise RuntimeError(
                    "SNOWFLAKE.TRUST_CENTER schema not found. Check edition, feature enablement, and TRUST_CENTER_VIEWER grants."
                )

            cur.execute("SELECT EVENT_ID, SEVERITY FROM SNOWFLAKE.TRUST_CENTER.FINDINGS LIMIT 1")
            sample_row = cur.fetchone()

        sample_message = (
            "Minimal read query succeeded and returned at least one row."
            if sample_row
            else "Minimal read query succeeded and returned zero rows."
        )
        return build_result(
            STATUS_PASS,
            "The role can see Snowflake Trust Center and execute a minimal read query.",
            details=session_details + ["Trust Center schema: found", sample_message],
        )
    except Exception as exc:
        result = classify_exception(exc, "trust_center_access")
        result["details"] = session_details + result.get("details", [])
        return result


def run_all_diagnostics(
    config: Dict[str, object],
    connect_fn: Optional[Callable[..., object]] = None,
    connector_version_override: Optional[str] = None,
    dns_resolver: Optional[Callable[[str], List[str]]] = None,
    tcp_checker: Optional[Callable[[str, int, int], None]] = None,
) -> Dict[str, Dict[str, object]]:
    """Run the complete diagnostics flow and close the connector session when done."""
    results: Dict[str, Dict[str, object]] = {}
    results["preflight"] = run_preflight_checks(
        config,
        connector_version_override=connector_version_override,
        dns_resolver=dns_resolver,
        tcp_checker=tcp_checker,
    )
    if results["preflight"]["status"] == STATUS_FAIL:
        results["login"] = skipped_result("Login check was skipped because preflight failed.")
        results["trust_center_access"] = skipped_result(
            "Trust Center access check was skipped because preflight failed."
        )
        return results

    login_result, conn = run_login_check(config, connect_fn=connect_fn)
    results["login"] = login_result
    if conn is None:
        results["trust_center_access"] = skipped_result(
            "Trust Center access check was skipped because login failed."
        )
        return results

    try:
        results["trust_center_access"] = run_access_checks(conn)
    finally:
        try:
            conn.close()
        except Exception:
            pass
    return results


def print_result(stage_name: str, result: Dict[str, object]) -> None:
    """Render one diagnostics block."""
    print(f"[{result['status']}] {stage_name}: {result['summary']}")
    for detail in result.get("details", []):
        print(f"  - {detail}")
    if result.get("actions"):
        print("  Next steps:")
        for action in result["actions"]:
            print(textwrap.indent(action, "    "))
    if result.get("raw_error"):
        print("  Raw error:")
        print(textwrap.indent(str(result["raw_error"]), "    "))
    print()


def print_final_summary(results: Dict[str, Dict[str, object]]) -> None:
    """Render a compact summary after the notebook finishes."""
    print("Final summary")
    print("=============")
    for stage_name in ["env", "preflight", "login", "trust_center_access"]:
        result = results.get(stage_name, skipped_result("Stage did not run."))
        print(f"- {stage_name}: {result['status']} - {result['summary']}")


In [ ]:
# Cell 4 - Load configuration
CONFIG, ENV_RESULT = prepare_config()
RESULTS = {"env": ENV_RESULT}
print_result("env", ENV_RESULT)

In [ ]:
# Cell 5 - Run diagnostics
if CONFIG is None:
    RESULTS["preflight"] = skipped_result("Preflight checks were skipped because configuration failed to load.")
    RESULTS["login"] = skipped_result("Login check was skipped because configuration failed to load.")
    RESULTS["trust_center_access"] = skipped_result("Trust Center access check was skipped because configuration failed to load.")
else:
    RESULTS.update(run_all_diagnostics(CONFIG))

print_result("preflight", RESULTS["preflight"])
print_result("login", RESULTS["login"])
print_result("trust_center_access", RESULTS["trust_center_access"])

In [ ]:
# Cell 6 - Final summary
print_final_summary(RESULTS)